# AutoDock Tutorial
### Notebook 02. Receptor Reconstruction by Comparative Modeling  

**Version 1.0.0 - April, 2026. Monterrey**


**Authors:** 
[Ana C. Murrieta ](https://orcid.org/0000-0002-7619-8880) and [Flavio F. Contreras-Torres](https://orcid.org/0000-0003-2375-131X). Tecnológico de Monterrey.



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/AutodockTutorial/blob/main/notebooks/pdb_structure_preparation.ipynb)

---


## Contents


This notebook begins with the definition of the computational environment required for sequence handling, structural parsing, and comparative modeling. It then defines the reconstruction target by restricting the modeling scope to the receptor region relevant for docking. The canonical receptor sequence is retrieved from [UniProt](https://www.uniprot.org/), and the experimental sequence is extracted from the cleaned crystallographic structure. Both sequences are subsequently compared to identify the crystallized fragment and the unresolved regions requiring reconstruction.

Based on this comparison, the modeling inputs are defined and converted into a template–target alignment in **PIR** format for [MODELLER](https://salilab.org/modeller/). Comparative modeling is then performed to generate multiple receptor models, which are evaluated using structural assessment scores. Finally, the best reconstructed model is examined by structural superposition against the crystal template, inspected for possible loop interference with the binding site, and renumbered according to the canonical UniProt sequence.


The activities are structured to be completed in approximately **60 to 120 minutes**. However, this is your personal workspace—we encourage you to add your own code cells, inspect additional PDB entries, and test alternative structural decisions as you build confidence with the workflow. The more you explore, the better prepared you'll be for the receptor preparation and docking steps that follow.


---

## How to work
1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourID_receptor_reconstruction.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

### MODELLER and PIR Format

[MODELLER](https://salilab.org/modeller/) is a software package for **comparative protein structure modeling**. In this notebook, it will be used to rebuild unresolved regions of the receptor based on the experimentally resolved structure and the corresponding target sequence.

> **Note:** This notebook assumes that **MODELLER** is already installed and properly configured in the active Python environment, including a valid license key. If MODELLER is not available, the modeling steps will not run.
>
> You can verify the installation with:
> ```python
> import importlib.util
> print(importlib.util.find_spec("modeller"))
> ```
>
> If the result is `None`, MODELLER is not installed in the active environment.


<br>

The **PIR format** is a structured plain-text sequence format used to represent one or more biological sequences, particularly in protein alignment and comparative modeling workflows. Each entry consists of a standardized header line (`>P1;identifier`), a descriptor line, followed by a descriptor line, and then the aligned sequence ending with `*`. This format allows that both sequence content and associated metadata to be encoded in a machine-readable form. A minimal example of a PIR entry looks like this:

```text
>P1;PPARG_target
sequence:PPARG_target::::::::
MGETLGDSPIDPESDSFTDTLSANISQEMTMV*
```

<br>


In general sequence databases, the second line may contain a free textual description of the sequence. However, in **MODELLER**, this line has a specific functional meaning: it is not a plain comment, but a structured record that tells the program whether the entry corresponds to a **template structure** (`structureX:`) or to a **target sequence** (`sequence:`), together with the residue range and chain identifier used for modeling.  

For this reason, the second line should not contain your local laptop path. Instead, it should contain only the template identifier or PDB filename required by MODELLER, provided that the working directory or `atom_files_directory` is already defined correctly. A clean example would be:

```text
>P1;5YCP_chainA_receptor
structureX:5YCP_chainA_receptor.pdb:204:A:477:A:::: 
LNPESADLRALAKHLYDSYIKSFPLTKAKARAILTGKTTDKSPFVIYDMNSLMMGEDKQSKEVAIRIFQGCQFRS
VEAVQEITEYAKSIPGFVNLDLNDQVTLLKYGVHEIIYTMLASLMNKDGVLISEGQGFMTREFLKSLRKPFGDFM
EPKFEFAVKFNALELDDSDLAIFIAVIILSGDRPGLLNVKPIEDIQDNLLQALELQLKLNHPESSQLFAKLLQKM
TDLRQIVTEHVQLLQVIKKTETDMSLHPLLQEIYKDLY*
```

<br>

In comparative modeling workflows, PIR files used with MODELLER should remain **informative, portable, and system-independent**, avoiding absolute local paths whenever possible, while providing the full structural and sequence metadata required for modeling. In this context, a PIR file does not merely store target and template sequences, but also encodes sequence identity, structural role, residue ranges, chain identifiers, and alignment gaps within a single formal representation, thereby establishing the explicit correspondence between an experimentally resolved template structure and its reference target sequence that is required for homology modeling.

---


### Step 0: The Computational Environment

Before starting receptor reconstruction, we must prepare a minimal computational environment for handling sequence files, structural inputs, sequence comparison, and comparative modeling tasks.

In this notebook, we rely on a small set of tools that support the reconstruction workflow:

* **`pathlib`**: for defining folders and file paths in a clean, platform-independent way.
* **`urllib`**: to retrieve the canonical receptor sequence from the **UniProt** database.
* **`Bio.SeqIO`, `Bio.Align`, and `Bio.PDB`** from **Biopython**: to read FASTA files, compare canonical and experimental sequences, and extract sequence information from the receptor structure.
* **`MODELLER`**: to perform comparative modeling and rebuild unresolved regions of the receptor structure from the template–target alignment.

This notebook can be executed either **locally** or in **Google Colab**, but these two environments should be considered separately. In a **local setup**, it is generally easier to manage folders, prepared input files, and external software such as MODELLER. In **Google Colab**, sequence handling and basic structure processing can be performed in the notebook, but external software dependencies may require additional setup and may not behave exactly as they do in a local environment.

> **Note:** Unlike Notebook 1, where the focus was structure retrieval and inspection, this notebook is centered on **sequence-guided receptor reconstruction**. For that reason, the key requirements here are access to the **experimental receptor structure**, the **canonical reference sequence**, and a working installation of **MODELLER** for downstream comparative modeling.
>
> **Practical note:** If you are working **locally**, MODELLER can be installed and configured directly in the active environment. If you are working in **Google Colab**, you should first verify whether the required dependencies can be installed and recognized properly before running the modeling steps.

## Step 1. Define the reconstruction target

First, we must define the exact scope of receptor reconstruction. The goal is not to rebuild the entire full-length protein, but only the region that is structurally and functionally relevant for the docking study.

At this stage, **the experimental crystal structure** should be used as the main reference, and **the canonical sequence** should serve only as a guide to identify unresolved or missing residues within the receptor region of interest. In practice, reconstruction should be limited to the crystallized domain required for docking, such as the ligand-binding domain (LBD), rather than forcing the modeling of distant termini or highly flexible segments with no clear structural support.

Special care should be taken with loops or terminal regions that are **absent from the crystal structure** because these regions are often intrinsically flexible and may not be necessary for ligand docking. Uncritical rebuilding of such segments may introduce unrealistic conformations, steric clashes, or artificial obstruction of the binding pocket.

Therefore, in this notebook, the reconstruction target will be restricted to the receptor region that is directly relevant for docking, while avoiding unnecessary modeling of flexible or poorly supported regions.

In [ ]:
# Step 1
from pathlib import Path

pdb_id = "5YCP"
target_chain = "A"

project_root = Path.cwd().parent
receptor_dir = project_root / "docking" / "prepared"
receptor_file = receptor_dir / f"{pdb_id}_chainA_receptor.pdb"

print(f"Receptor file selected: {receptor_file.name}")
print(f"Target chain: {target_chain}")

print("\nReconstruction target:")
print("- Focus only on the receptor region relevant for docking")
print("- Do not force full-length modeling unless structurally justified")
print("- Avoid unnecessary reconstruction of flexible termini or unsupported loops")

## Step 2. Retrieve the canonical reference sequence

The next step is to obtain the canonical amino acid sequence of the receptor, which will be used as the biological reference for comparative modeling. This sequence represents the curated full-length protein record and provides the framework needed to identify unresolved or missing regions in the experimental crystal structure.

Remember that the canonical sequence is not used to replace the crystal structure, but to guide the reconstruction of residues that were not resolved during structure determination. The experimental structure remains the main structural template, while the canonical sequence serves as the reference target for sequence comparison and downstream modeling.

Thus, the canonical receptor sequence will be retrieved from the **UniProt** database and prepared for alignment against the receptor sequence extracted from the crystal structure.

In [ ]:
# Step 2
from pathlib import Path
from urllib.request import urlopen

uniprot_id = "P37231"
project_root = Path.cwd().parent
sequence_dir = project_root / "docking" / "sequence"
sequence_dir.mkdir(parents=True, exist_ok=True)

fasta_file = sequence_dir / f"{uniprot_id}.fasta"
fasta_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"

with urlopen(fasta_url) as response:
    fasta_text = response.read().decode("utf-8")

with open(fasta_file, "w") as f:
    f.write(fasta_text)

print(f"Canonical sequence retrieved successfully: {uniprot_id}")
print(f"Saved to: {fasta_file}")

print("\nFASTA preview:")
print("\n".join(fasta_text.splitlines()[:4]))

> **Output note:**  
> The canonical receptor sequence is retrieved from **UniProt** and saved in **FASTA** format. This file, for instance, `P37231.fasta`, will serve as the biological reference sequence for the next steps, where it will be compared against the receptor sequence extracted from the experimental crystal structure.

## Step 3. Extract the receptor sequence from the experimental structure

The next step is to extract the amino acid sequence that is actually present in the experimental receptor structure. This sequence corresponds only to the residues resolved in the crystallographic model and therefore reflects the structural fragment available as template for comparative modeling.

The purpose of this step is to generate a sequence representation of the receptor directly from the prepared PDB structure. This experimental sequence will later be compared with the canonical UniProt sequence in order to identify missing segments, unresolved residues, or truncations relevant for receptor reconstruction.

Thus, the receptor sequence will be extracted from the cleaned receptor structure and saved in FASTA format for downstream alignment.

In [ ]:
# Step 3
from pathlib import Path
from Bio.PDB import PDBParser, PPBuilder

pdb_id = "5YCP"
target_chain = "A"

project_root = Path.cwd().parent
receptor_dir = project_root / "docking" / "prepared"
sequence_dir = project_root / "docking" / "sequence"

receptor_file = receptor_dir / f"{pdb_id}_chainA_receptor.pdb"
output_fasta = sequence_dir / f"{pdb_id}_chain{target_chain}_sequence.fasta"

parser = PDBParser(QUIET=True)
structure = parser.get_structure(pdb_id, receptor_file)

model = structure[0]
chain = model[target_chain]

ppb = PPBuilder()
peptides = ppb.build_peptides(chain)
sequence = "".join(str(peptide.get_sequence()) for peptide in peptides)

with open(output_fasta, "w") as f:
    f.write(f">{pdb_id}_chain{target_chain}\n")
    f.write(sequence + "\n")

print(f"Experimental receptor sequence extracted successfully from chain {target_chain}")
print(f"Saved to: {output_fasta}")
print(f"Sequence length: {len(sequence)} aa")

print("\nSequence preview:")
print(sequence[:80])

> **Output note:**  
> The receptor sequence is extracted from the experimental structure and saved in **FASTA** format. In general, the sequence obtained from a crystal structure represents only the portion of the protein that was resolved experimentally, so it may be shorter than the corresponding canonical reference sequence. This extracted sequence will be used in the next step for alignment against the canonical sequence in order to identify missing or unresolved regions relevant for receptor reconstruction.

## Step 4. Compare experimental and canonical sequences

Once both sequences have been prepared, the next step is to compare the receptor sequence extracted from the experimental structure against the canonical reference sequence. This comparison allows us to determine which parts of the protein are represented in the crystal structure and which regions are absent, truncated, or unresolved.

The purpose of this step is not yet to generate the final **MODELLER** alignment file, but to establish the biological correspondence between the experimental structure and the full reference sequence. This comparison provides the basis for defining the reconstruction target and for identifying which missing regions, if any, should be modeled in later steps.

The experimental and canonical sequences will be aligned to identify their overlap and to detect missing or unresolved segments relevant for receptor reconstruction.

In [ ]:
# Step 4
from pathlib import Path
from Bio import SeqIO
from Bio.Align import PairwiseAligner

project_root = Path.cwd().parent
sequence_dir = project_root / "docking" / "sequence"

canonical_fasta = sequence_dir / "P37231.fasta"
experimental_fasta = sequence_dir / "5YCP_chainA_sequence.fasta"
alignment_file = sequence_dir / "sequence_comparison.txt"

canonical_record = SeqIO.read(canonical_fasta, "fasta")
experimental_record = SeqIO.read(experimental_fasta, "fasta")

canonical_seq = str(canonical_record.seq)
experimental_seq = str(experimental_record.seq)

aligner = PairwiseAligner()
aligner.mode = "local"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

best_alignment = aligner.align(canonical_seq, experimental_seq)[0]

with open(alignment_file, "w") as f:
    f.write(str(best_alignment))

print("Sequence comparison completed successfully")
print(f"Canonical sequence length:    {len(canonical_seq)} aa")
print(f"Experimental sequence length: {len(experimental_seq)} aa")
print(f"Alignment score:             {best_alignment.score}")
print(f"Saved to: {alignment_file}")

print("\nAlignment preview:\n")
print(best_alignment)

> **Output note:**  
> This step compares the two sequences obtained in the previous stages: the **canonical reference sequence from UniProt** and the **experimental sequence extracted from the PDB structure**. The comparison is performed through a **sequence alignment**, which shows which portion of the canonical protein is represented in the experimental structure and helps identify sequence gaps associated with unresolved or missing residues.  
>
> In this example, the experimental receptor sequence (263 aa) aligns to the C-terminal region of the canonical PPARG sequence, spanning approximately residues **231 to 505**. This confirms that the crystal structure represents a truncated fragment of the receptor rather than the full-length protein. The alignment also shows small internal gaps, consistent with unresolved residues within the crystallized region. This information is sufficient to define the portion of the receptor that can be used as structural template for comparative modeling.  
>
> In general, this step defines the structural overlap between the experimentally resolved fragment and the full biological reference, providing the basis for deciding what should be modeled in later stages.

## Step 5. Define the modeling inputs

The next step is to define the inputs that will be used for comparative modeling. The goal is to decide which structural region will serve as template, which sequence will be used as target, and whether the reconstruction should be limited to the crystallized receptor fragment relevant for docking.

This step is important because comparative modeling should not be applied indiscriminately to the entire full-length canonical sequence if only a specific crystallized domain is structurally relevant for the study. In many docking workflows, only the experimentally resolved receptor region, such as the ligand-binding domain, is needed as template for rebuilding missing residues.

In this cell, the modeling inputs will therefore be defined by combining the cleaned experimental receptor structure with the corresponding target sequence region supported by the sequence comparison performed in the previous step.

In [ ]:
# Step 5
from pathlib import Path
from Bio import SeqIO

pdb_id = "5YCP"
target_chain = "A"
uniprot_id = "P37231"

project_root = Path.cwd().parent
prepared_dir = project_root / "docking" / "prepared"
sequence_dir = project_root / "docking" / "sequence"
modeling_dir = project_root / "docking" / "homology"
modeling_dir.mkdir(parents=True, exist_ok=True)

template_pdb = prepared_dir / f"{pdb_id}_chainA_receptor.pdb"
canonical_fasta = sequence_dir / f"{uniprot_id}.fasta"

# Based on the sequence comparison from Step 4
target_start = 231
target_end = 505

canonical_record = SeqIO.read(canonical_fasta, "fasta")
canonical_seq = str(canonical_record.seq)
target_seq = canonical_seq[target_start - 1:target_end]

target_fasta = modeling_dir / f"{uniprot_id}_{target_start}_{target_end}.fasta"

with open(target_fasta, "w") as f:
    f.write(f">{uniprot_id}_{target_start}_{target_end}\n")
    f.write(target_seq + "\n")

print("Modeling inputs defined successfully")
print(f"Template structure: {template_pdb.name}")
print(f"Target sequence:    {target_fasta.name}")
print(f"Target region:      {target_start}-{target_end}")
print(f"Target length:      {len(target_seq)} aa")

> **Output note:**  
> In this step, the inputs required for comparative modeling are defined from the sequence comparison obtained previously. The cleaned receptor structure is selected as the **template**, and the corresponding region of the canonical reference sequence is extracted as the **target** for modeling.  
>
> In this example, the template structure is `5YCP_chainA_receptor.pdb`, and the target sequence was saved as `P37231_231_505.fasta`, corresponding to residues **231 to 505** of the canonical PPARG sequence, with a total length of **275 aa**.  
>
> This means that the workflow is no longer using the full canonical protein sequence as modeling input, but only the region supported by the structural overlap detected in the previous alignment. In practical terms, this step converts the biological interpretation of the alignment into concrete inputs that can be used in the next stage to prepare the MODELLER alignment file.

## Step 6. Generate the MODELLER alignment file

After defining the structural template and the target sequence region for reconstruction, the next step is to prepare the alignment file required by **MODELLER**. This file is not simply a standard sequence alignment, but a specific input that connects the experimental structure used as template with the target sequence to be modeled.

The purpose is to translate the previously defined modeling inputs into a format that MODELLER can interpret correctly. The resulting file must preserve the residue correspondence between template and target, while explicitly representing sequence gaps associated with unresolved or missing residues.

In this step, the **MODELLER alignment file** will be generated from the cleaned receptor structure and the selected target sequence region, using the sequence overlap established in the previous steps as the basis for comparative modeling.

In [ ]:
# Step 6 (hybrid correction, adjusted)
from pathlib import Path
from Bio import SeqIO
from Bio.Align import PairwiseAligner
from modeller import Environ, Model, Alignment, log

pdb_id = "5YCP"
target_chain = "A"
template_id = "5YCP_chainA_receptor"
target_id = "PPARG_P37231_231_505"

# Template PDB numbering detected by MODELLER
template_start = 204
template_end = 477

# Target canonical numbering
target_start = 231
target_end = 505

print("Locating project root...", flush=True)

current = Path.cwd().resolve()
project_root = None

for candidate in [current] + list(current.parents):
    if (candidate / "docking" / "prepared").exists() and \
       (candidate / "docking" / "sequence").exists() and \
       (candidate / "docking" / "homology").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("Project root could not be located automatically.")

prepared_dir = project_root / "docking" / "prepared"
sequence_dir = project_root / "docking" / "sequence"
modeling_dir = project_root / "docking" / "homology"

template_pdb = prepared_dir / f"{pdb_id}_chainA_receptor.pdb"
target_fasta = modeling_dir / "P37231_231_505.fasta"
ali_file = modeling_dir / "seq_5YCP.ali"
tmp_template_seq = modeling_dir / "template_from_modeller.seq"

print(f"Project root found: {project_root}", flush=True)
print("Extracting template sequence with MODELLER...", flush=True)

# Reduce MODELLER verbosity
log.none()

env = Environ()
env.io.atom_files_directory = [str(prepared_dir), str(modeling_dir)]

aln_tmp = Alignment(env)
mdl = Model(env, file=str(template_pdb))
aln_tmp.append_model(mdl, align_codes=template_id, atom_files=str(template_pdb))
aln_tmp.write(file=str(tmp_template_seq), alignment_format="PIR")

print("Template sequence extracted.", flush=True)
print("Parsing MODELLER PIR sequence...", flush=True)

def read_pir_sequence(pir_file):
    lines = Path(pir_file).read_text().splitlines()
    seq_lines = []
    started = False
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith(">P1;"):
            if started:
                break
            started = True
            continue
        if started and line.startswith(("structureX:", "sequence:")):
            continue
        if started:
            seq_lines.append(line)
    return "".join(seq_lines).replace("*", "").replace("/", "").replace(" ", "")

template_seq = read_pir_sequence(tmp_template_seq)
target_seq = str(SeqIO.read(target_fasta, "fasta").seq)

print(f"Template sequence length: {len(template_seq)}", flush=True)
print(f"Target sequence length:   {len(target_seq)}", flush=True)
print("Running global sequence alignment...", flush=True)

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

alignment = aligner.align(target_seq, template_seq)[0]

target_aligned = []
template_aligned = []

target_pos = 0
template_pos = 0

for t_block, e_block in zip(alignment.aligned[0], alignment.aligned[1]):
    t_start, t_end = t_block
    e_start, e_end = e_block

    if t_start > target_pos:
        target_aligned.append(target_seq[target_pos:t_start])
        template_aligned.append("-" * (t_start - target_pos))

    if e_start > template_pos:
        target_aligned.append("-" * (e_start - template_pos))
        template_aligned.append(template_seq[template_pos:e_start])

    target_aligned.append(target_seq[t_start:t_end])
    template_aligned.append(template_seq[e_start:e_end])

    target_pos = t_end
    template_pos = e_end

if target_pos < len(target_seq):
    target_aligned.append(target_seq[target_pos:])
    template_aligned.append("-" * (len(target_seq) - target_pos))

if template_pos < len(template_seq):
    target_aligned.append("-" * (len(template_seq) - template_pos))
    template_aligned.append(template_seq[template_pos:])

target_aligned = "".join(target_aligned)
template_aligned = "".join(template_aligned)

print("Writing final MODELLER alignment file...", flush=True)

with open(ali_file, "w") as f:
    f.write(f">P1;{template_id}\n")
    f.write(f"structureX:{template_pdb.stem}:{template_start}:{target_chain}:{template_end}:{target_chain}::::\n")
    f.write(template_aligned + "*\n\n")

    f.write(f">P1;{target_id}\n")
    f.write(f"sequence:{target_id}:{target_start}:{target_chain}:{target_end}:{target_chain}::::\n")
    f.write(target_aligned + "*\n")

print("MODELLER alignment file generated successfully", flush=True)
print(f"Saved to: {ali_file}", flush=True)
print(f"Template identifier: {template_id}", flush=True)
print(f"Target identifier:   {target_id}", flush=True)
print(f"Template segment:    {template_start}-{template_end}", flush=True)
print(f"Target segment:      {target_start}-{target_end}", flush=True)
print(f"Template aligned length: {len(template_aligned)}", flush=True)
print(f"Target aligned length:   {len(target_aligned)}", flush=True)

> **Output note:**  
> In this step, the modeling inputs are converted into the **PIR alignment file** required by **MODELLER**. The target sequence corresponded to the selected canonical region of the receptor, while the template sequence is extracted directly from the cleaned receptor structure so that the alignment would remain fully consistent with the PDB file used for comparative modeling.  
>
> In this example, the alignment file was saved as `seq_5YCP.ali`, using `5YCP_chainA_receptor` as the template identifier and `PPARG_P37231_231_505` as the target identifier. The template sequence extracted from the structure has a length of **263 residues**, the target sequence has a length of **275 residues**, and the final aligned template length reported was **275 residues**, indicating that both sequences were successfully placed into a common alignment framework for modeling.  
>
> An important detail in this example is that the **template structure numbering** and the **canonical target numbering** are not the same. The receptor structure used as template spans residues **204–477** in chain A, whereas the canonical target region corresponds to residues **231–505**. The alignment file preserves this distinction explicitly, which is essential for MODELLER to interpret the template structure and the target sequence correctly.  
>
> In practical terms, this step produces the structured alignment file that will be used directly by **MODELLER** in the next stage to build reconstructed receptor models.

## Step 7. Build and evaluate receptor models by comparative modeling

Once the **MODELLER alignment file** has been prepared, the next step is to generate reconstructed receptor models by comparative modeling. At this stage, the cleaned experimental structure is used as the **template**, while the selected canonical sequence region is used as the **target** to rebuild unresolved residues or segments that were not fully resolved in the crystallographic model.

The objective of this step is not to produce a single model immediately, but to generate **multiple candidate structures** that can be compared and ranked. This is important because comparative modeling can produce alternative conformations, especially in reconstructed loops or flexible regions, and these differences may affect the structural quality of the model or its suitability for downstream docking.

In this implementation, **MODELLER** generates several receptor models from the template–target alignment and evaluates them using structural scoring functions, including **molpdf**, **DOPE**, and **GA341**. These scores provide a first criterion for selecting the most reliable model, with particular emphasis on **DOPE** as the main ranking metric.

At the end of this step, the notebook identifies the best candidate model according to the selected assessment criteria. That model will then be inspected structurally in the following stage before continuing with receptor preparation for docking.

In [ ]:
# Step 7 (revised: with DOPE and GA341)
from pathlib import Path
import os

try:
    from modeller import Environ, log
    from modeller.automodel import AutoModel, assess
except ImportError as e:
    raise ImportError(
        f"MODELLER is available, but one of the required modeling modules "
        f"could not be imported: {e}"
    )

pdb_id = "5YCP"
template_id = "5YCP_chainA_receptor"
target_id = "PPARG_P37231_231_505"
n_models = 3

print("Locating project root...", flush=True)

current = Path.cwd().resolve()
project_root = None

for candidate in [current] + list(current.parents):
    if (candidate / "docking" / "prepared").exists() and \
       (candidate / "docking" / "homology").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("Project root could not be located automatically.")

prepared_dir = project_root / "docking" / "prepared"
modeling_dir = project_root / "docking" / "homology"

ali_file = modeling_dir / "seq_5YCP.ali"
template_pdb = prepared_dir / f"{pdb_id}_chainA_receptor.pdb"

if not ali_file.exists():
    raise FileNotFoundError(f"Alignment file not found: {ali_file}")

if not template_pdb.exists():
    raise FileNotFoundError(f"Template PDB not found: {template_pdb}")

print(f"Project root found: {project_root}", flush=True)
print("Preparing comparative modeling run...", flush=True)

os.chdir(modeling_dir)
log.none()

env = Environ()
env.io.atom_files_directory = [str(modeling_dir), str(prepared_dir)]

print("Building models with MODELLER...", flush=True)

a = AutoModel(
    env,
    alnfile=ali_file.name,
    knowns=template_id,
    sequence=target_id,
    assess_methods=(assess.DOPE, assess.GA341)
)

a.starting_model = 1
a.ending_model = n_models
a.make()

print("Comparative modeling finished successfully", flush=True)
print(f"Alignment file used: {ali_file.name}", flush=True)
print(f"Template used:       {template_pdb.name}", flush=True)
print(f"Target used:         {target_id}", flush=True)
print(f"Number of models:    {n_models}", flush=True)
print(f"Output directory:    {modeling_dir}", flush=True)

# Optional: rank models explicitly by DOPE
ok_models = [m for m in a.outputs if m["failure"] is None]

if ok_models:
    ok_models.sort(key=lambda x: x["DOPE score"])
    best_model = ok_models[0]

    print("\nBest model by DOPE:", flush=True)
    print(f"Filename:    {best_model['name']}", flush=True)
    print(f"molpdf:      {best_model['molpdf']}", flush=True)
    print(f"DOPE score:  {best_model['DOPE score']}", flush=True)
    
    ga341_value = best_model["GA341 score"]
    if isinstance(ga341_value, (list, tuple)):
        ga341_value = ga341_value[0]

    print(f"GA341 score: {ga341_value}", flush=True)
else:
    print("No successful models were available for ranking.", flush=True)




> **Output note:**  
> In this step, **MODELLER** generates multiple receptor models from the template–target alignment and evaluates them using **molpdf**, **DOPE**, and **GA341** scores. This confirms that the alignment file was interpreted correctly and that comparative modeling could be completed without sequence or template inconsistencies.  
>
> In this example, three models were produced: `PPARG_P37231_231_505.B99990001.pdb`, `PPARG_P37231_231_505.B99990002.pdb`, and `PPARG_P37231_231_505.B99990003.pdb`. Among them, model `B99990002` showed the lowest **molpdf** value and the most favorable **DOPE score**, making it the best candidate under the current ranking criteria. All three models also showed a **GA341 score of 1.00000**, supporting their overall reliability within the comparative modeling framework.  
>
> In practical terms, this step yields a small set of reconstructed receptor structures and provides an initial objective basis for selecting the best model. The top-ranked model can now be inspected structurally against the experimental template before continuing with receptor preparation for docking.

## Step 8. Evaluate and select the best reconstructed model

The next step is to identify which reconstructed structure is most suitable for downstream docking. Model selection should not rely only on comparative modeling scores, but also on structural inspection of the reconstructed regions and their relationship to the ligand-binding site.

Thus, the best-ranked model is evaluated against the cleaned crystallographic structure by **structural superposition** and, when appropriate, by **RMSD comparison**. This allows us to verify whether the reconstructed model preserves the overall fold of the experimental receptor while introducing only local changes in unresolved regions.

Special attention should be given to the **reconstructed loops or missing segments**, since these regions may adopt alternative conformations that invade, occlude, or distort the ligand-binding cavity. Even a model with favorable comparative modeling scores may be unsuitable for docking if a rebuilt loop occupies the orthosteric site or alters the accessibility of the pocket.

Finally, once the most appropriate reconstructed model has been selected, its residue numbering should be **renumbered to match the canonical UniProt sequence**. This ensures consistency between the final structural model and the biological reference sequence, facilitating interpretation, visualization, and subsequent docking analysis.

In practical terms, this step combines **score-based selection**, **structural comparison**, **binding-site inspection**, and **final renumbering** to define the receptor model that will be carried forward for docking preparation.

In [ ]:
# Step 8
from pathlib import Path
from Bio.PDB import PDBParser, Superimposer, PDBIO, Select
from Bio.PDB.Polypeptide import is_aa
from Bio.Align import PairwiseAligner
from Bio.SeqUtils import seq1

pdb_id = "5YCP"
template_chain_id = "A"
model_chain_id = "A"

best_model_name = "PPARG_P37231_231_505.B99990002.pdb"
template_name = f"{pdb_id}_chainA_receptor.pdb"
ligand_name = f"{pdb_id}_BRL_ligand.pdb"

target_start = 231  # canonical UniProt numbering start for the selected region

print("Locating project root...", flush=True)

current = Path.cwd().resolve()
project_root = None

for candidate in [current] + list(current.parents):
    if (candidate / "docking" / "prepared").exists() and \
       (candidate / "docking" / "homology").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("Project root could not be located automatically.")

prepared_dir = project_root / "docking" / "prepared"
modeling_dir = project_root / "docking" / "homology"

template_file = prepared_dir / template_name
ligand_file = prepared_dir / ligand_name
best_model_file = modeling_dir / best_model_name

if not template_file.exists():
    raise FileNotFoundError(f"Template file not found: {template_file}")
if not best_model_file.exists():
    raise FileNotFoundError(f"Best model file not found: {best_model_file}")
if not ligand_file.exists():
    raise FileNotFoundError(f"Ligand file not found: {ligand_file}")

print(f"Template file: {template_file.name}", flush=True)
print(f"Best model:    {best_model_file.name}", flush=True)
print(f"Ligand file:   {ligand_file.name}", flush=True)

parser = PDBParser(QUIET=True)

template_structure = parser.get_structure("template", template_file)
model_structure = parser.get_structure("model", best_model_file)
ligand_structure = parser.get_structure("ligand", ligand_file)

template_chain = template_structure[0][template_chain_id]
model_chain = model_structure[0][model_chain_id]

def get_protein_residues(chain):
    residues = []
    for res in chain:
        if is_aa(res, standard=True):
            residues.append(res)
    return residues

def residue_to_one_letter(residue):
    return seq1(residue.resname)

template_residues = get_protein_residues(template_chain)
model_residues = get_protein_residues(model_chain)

template_seq = "".join(residue_to_one_letter(r) for r in template_residues)
model_seq = "".join(residue_to_one_letter(r) for r in model_residues)

print(f"Template residues: {len(template_residues)}", flush=True)
print(f"Model residues:    {len(model_residues)}", flush=True)

# --- Sequence-based mapping for superposition ---
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

alignment = aligner.align(template_seq, model_seq)[0]

template_atoms = []
model_atoms = []

model_only_residue_indices = []
template_pos = 0
model_pos = 0

for t_block, m_block in zip(alignment.aligned[0], alignment.aligned[1]):
    t_start, t_end = t_block
    m_start, m_end = m_block

    # residues present only in model before the next aligned block
    if m_start > model_pos:
        model_only_residue_indices.extend(range(model_pos, m_start))

    overlap = min(t_end - t_start, m_end - m_start)

    for i in range(overlap):
        t_res = template_residues[t_start + i]
        m_res = model_residues[m_start + i]

        if "CA" in t_res and "CA" in m_res:
            template_atoms.append(t_res["CA"])
            model_atoms.append(m_res["CA"])

    template_pos = t_end
    model_pos = m_end

# residues present only in model after the last aligned block
if model_pos < len(model_residues):
    model_only_residue_indices.extend(range(model_pos, len(model_residues)))

if len(template_atoms) == 0 or len(model_atoms) == 0:
    raise ValueError("No matched CA atoms were found for superposition.")

# --- Superimpose best model onto template ---
sup = Superimposer()
sup.set_atoms(template_atoms, model_atoms)
sup.apply(model_structure.get_atoms())
rmsd = sup.rms

print(f"Matched CA atoms used for superposition: {len(template_atoms)}", flush=True)
print(f"Cα RMSD (model vs template): {rmsd:.3f} Å", flush=True)

# --- Evaluate possible loop invasion into the binding site ---
ligand_atoms = list(ligand_structure.get_atoms())

if len(ligand_atoms) == 0:
    raise ValueError("No ligand atoms were found in the ligand file.")

model_only_residues = [model_residues[i] for i in model_only_residue_indices]

def min_distance_between_residue_and_ligand(residue, ligand_atoms):
    min_dist = float("inf")
    for atom in residue.get_atoms():
        for lig_atom in ligand_atoms:
            dist = atom - lig_atom
            if dist < min_dist:
                min_dist = dist
    return min_dist

loop_contacts = []
for res in model_only_residues:
    dmin = min_distance_between_residue_and_ligand(res, ligand_atoms)
    loop_contacts.append((res.id[1], res.resname, dmin))

loop_contacts_sorted = sorted(loop_contacts, key=lambda x: x[2])

print(f"Modeled-only residues detected: {len(model_only_residues)}", flush=True)

if loop_contacts_sorted:
    closest_resnum, closest_resname, closest_dist = loop_contacts_sorted[0]
    print(
        f"Closest modeled residue to ligand: {closest_resname} {closest_resnum} "
        f"at {closest_dist:.2f} Å",
        flush=True
    )

    if closest_dist < 2.5:
        print("Warning: reconstructed region may invade the binding site.", flush=True)
    elif closest_dist < 4.0:
        print("Caution: reconstructed region is close to the binding site.", flush=True)
    else:
        print("No obvious binding-site invasion detected from reconstructed residues.", flush=True)
else:
    print("No modeled-only residues were detected for loop invasion analysis.", flush=True)

# --- Renumber final selected model to UniProt numbering (two-pass safe renumbering) ---
class ProteinSelect(Select):
    def accept_residue(self, residue):
        return is_aa(residue, standard=True)

renumbered_model = parser.get_structure("renumbered_model", best_model_file)
renumbered_chain = renumbered_model[0][model_chain_id]
renumbered_residues = [r for r in renumbered_chain if is_aa(r, standard=True)]

# Pass 1: assign temporary residue numbers to avoid ID collisions
tmp_number = -1000
for residue in renumbered_residues:
    residue.id = (" ", tmp_number, " ")
    tmp_number += 1

# Pass 2: assign final UniProt numbering
new_number = target_start
for residue in renumbered_residues:
    residue.id = (" ", new_number, " ")
    new_number += 1

renumbered_file = modeling_dir / best_model_name.replace(".pdb", "_renum_uniprot.pdb")

io = PDBIO()
io.set_structure(renumbered_model)
io.save(str(renumbered_file), select=ProteinSelect())

print(f"Renumbered model saved to: {renumbered_file}", flush=True)
print("Step 8 completed successfully.", flush=True)

> **Output note:**  
> In this final step, the top-ranked reconstructed model is evaluated structurally against the cleaned crystallographic template. 
>
> The comparison showed that the model preserves the overall receptor fold, with **263 matched Cα atoms** and a **Cα RMSD of 0.212 Å**, indicating very close agreement between the reconstructed structure and the experimental receptor.  
>
> The reconstructed regions were also inspected relative to the crystallographic ligand in order to detect possible **binding-site invasion** by modeled loops or inserted residues. In this example, **12 modeled-only residues** were identified, and the closest of them was **ILE 60**, located **8.59 Å** from the ligand. This distance does not suggest occlusion of the binding cavity, so no obvious invasion of the orthosteric site was detected.  
>
> Finally, the selected model was saved as a **renumbered PDB file** consistent with the canonical UniProt reference sequence: `PPARG_P37231_231_505.B99990002_renum_uniprot.pdb`.  
>
> In practical terms, this step confirms that the selected reconstructed model is structurally consistent with the experimental template, does not show obvious loop interference with the ligand-binding site, and is now available in a final renumbered form for downstream receptor preparation and docking studies.

---


### Summary and Next Step

You have successfully reconstructed and selected a receptor model suitable for downstream docking studies.

**Current State**  
We now have:

- the canonical receptor sequence retrieved from **UniProt**
- the experimental receptor sequence extracted from the cleaned crystallographic structure
- a validated sequence comparison defining the crystallized region and unresolved residues
- a **MODELLER** alignment file prepared for comparative modeling:
  - `seq_5YCP.ali`
- three reconstructed receptor models generated and evaluated by **molpdf**, **DOPE**, and **GA341**
- a structurally validated best model selected after superposition against the crystal template and inspection of possible loop interference with the binding site
- a final renumbered receptor model consistent with the canonical UniProt sequence:
  - `PPARG_P37231_231_505.B99990002_renum_uniprot.pdb`

<br>

**Next Step**  
In the next notebook, we will prepare the selected reconstructed receptor for molecular docking. We will use `PPARG_P37231_231_505.B99990002_renum_uniprot.pdb` as the working receptor structure, verify its final structural consistency, define the appropriate chemical state by adding hydrogens and assigning protonation, and prepare the ligand under compatible conditions. The receptor and ligand will then be converted to the docking format required by the selected software, and the docking search space will be defined using the crystallographic binding site as reference. A final validation step will confirm that the receptor is complete, the binding cavity remains accessible, the reconstructed loop does not interfere with the pocket, and the docking box adequately covers the target site.

---

### Exercises

Now that you have completed the receptor reconstruction workflow, apply the same strategy to additional protein structures in which the crystallographic receptor may contain unresolved residues or missing segments.

These exercises are intended to help you practice how to define the reconstruction target, compare the experimental structure against the canonical sequence, generate a valid MODELLER alignment, and evaluate whether the reconstructed model is structurally suitable for docking.

> **Important:** Not all crystal structures require reconstruction to the same extent.  
> Some receptors are nearly complete and may only contain minor unresolved loops, whereas others represent only a truncated functional domain.  
> For that reason, always determine first whether reconstruction is truly necessary, which region should be modeled, and whether the rebuilt segments remain compatible with docking in the binding site of interest.

---

**Exercise 1: Repeat the reconstruction workflow with a different receptor structure**  
Select a new PDB entry between  `"6MD4"`, `"2POB"`, `"6K0T"`. Take your choice and repeat the full workflow from canonical sequence retrieval to final model evaluation.

For your selected case, determine:

1. which portion of the canonical sequence is represented in the experimental structure,
2. whether internal gaps or missing regions are present,
3. which receptor segment should be modeled for docking,
4. how many comparative models are generated,
5. and which model is selected as the best candidate based on DOPE and structural inspection.

Summarize your conclusions in a short paragraph.

---

**Exercise 2: Compare two reconstruction scenarios**  
Apply the workflow to two different receptor structures and compare the outcomes.

Discuss whether:

1. both structures required reconstruction,
2. the same receptor region was modeled in both cases,
3. the reconstructed segments were close to the ligand-binding site,
4. the best models showed similar agreement with the crystallographic templates,
5. and one structure appeared more suitable for docking than the other.

Write a brief comparison emphasizing the structural factors that influenced model selection.

---

**Exercise 3: Evaluate the impact of reconstruction on docking readiness**  
Using one reconstructed receptor model, inspect whether the rebuilt residues improve the structural completeness of the receptor without compromising the accessibility of the binding cavity.

In your analysis, address:

1. the RMSD between the reconstructed model and the crystal template,
2. the location of the reconstructed residues relative to the ligand-binding site,
3. whether any rebuilt loop appears to interfere with the pocket,
4. and whether the final renumbered model can be considered ready for receptor preparation and docking.

Conclude with a short statement indicating whether you would keep or reject that reconstructed model for downstream docking studies.

---

### Outlook

Congratulations! You have completed a comparative modeling workflow for receptor reconstruction and structural validation. You now possess the skills to:

* retrieve the canonical receptor sequence from **UniProt**
* extract the experimental sequence directly from a crystallographic receptor structure
* compare canonical and experimental sequences to define the region requiring reconstruction
* generate a valid **MODELLER** alignment file in PIR format
* build and evaluate multiple receptor models using **molpdf**, **DOPE**, and **GA341**
* assess reconstructed models by structural superposition, **RMSD**, and inspection of possible loop interference with the ligand-binding site
* produce a final receptor structure renumbered according to the canonical UniProt sequence and suitable for downstream docking preparation

See you in the next lesson!

---

### License  
The content of this tutorial itself is licensed under the terms and conditions of the [Creative Commons Attribution (CC BY 4.0) license](https://creativecommons.org/licenses/by/4.0/legalcode.en), and the underlying source code used to format and display that content is licensed under the [MIT license](https://github.com/NanoBiostructuresRG/AutodockTutorial/blob/main/LICENSE). See the LICENSE files for full details.

### Attribution
If you use or adapt this material, please provide appropriate credit to the original authors, [ACM](https://orcid.org/0000-0002-7619-8880) and [FFCT](https://orcid.org/0000-0003-2375-131X), as well as to the repository: [https://github.com/NanoBiostructuresRG](https://github.com/NanoBiostructuresRG).